## But

Vérifier la validation du pipeline prévu dans AR1

## Principe général

* Historique complet **1959–2025** sur **UNRATE**.
* Données consommées exclusivement via **Feast**.
* Modèle Baseline et univarié.
* Backtesting temporel minimal avec prévisions ponctuelles et intervalles de prédiction conformes (95 %).
* Comparer AR1 et ARp

## Résultat
cf la fin

# Package

In [20]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla
# ----------------------------
from statsforecast import StatsForecast
from statsforecast.models import AutoRegressive
from statsforecast.utils import ConformalIntervals
from utilsforecast.plotting import plot_series

# Importation des données

In [21]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Dictionnaire de modèle

In [22]:
from statsforecast.models import AutoRegressive

SF_MODELS = {
    "AR_1":   lambda: AutoRegressive(lags=1)}

# Backtesting

In [23]:
from mlforecast.utils import PredictionIntervals

def run_backtesting_h12_simple(
    mlf,
    ts,
    *,
    h=12,
    step_size=12,
    partitions=4,
    pi_windows=3,
    levels=[95],
):
    """
    Simple backtesting:
    - horizon h (default: 12 months)
    - step_size between cutoffs (default: 12 months)
    - few partitions (default: 4)
    - conformal prediction intervals
    """

    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
    )

    return bkt_df

# Run 

In [24]:

# ----------------------------
# Config
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"

FEATURE_REFS = ["stationary_value:value"]  # y uniquement

H = 12
STEP_SIZE = 12
TEST_START = "1990-01-01"
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

AR_LAGS = 12  # AR

In [25]:
# ----------------------------
# 1) entity_df
# ----------------------------
dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

In [26]:
# ----------------------------
# 2) Feast -> ts (format StatsForecast)
# ----------------------------
ts_raw = load_features_from_feast(entity_df=entity_df, feature_refs=FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


In [27]:
# ----------------------------
# 3) StatsForecast (p auto)
# ----------------------------
from statsforecast.models import AutoARIMA

sf = StatsForecast(
    models=[
        AutoARIMA(
            d=0, D=0,
            max_p=12, max_q=0,   # AR(p) only
            seasonal=False,
            stationary=True,
        )
    ],
    freq=FREQ,
)

In [28]:

# ----------------------------
# 4) Cross-validation + Conformal intervals
# ----------------------------
STEP_SIZE = 36  # <-- update / refit tous les 36 mois (expanding)
ci = ConformalIntervals(h=H, n_windows=PI_WINDOWS)

test_start_ts = pd.Timestamp(TEST_START, tz="UTC")
ds_sorted = ts["ds"].sort_values().reset_index(drop=True)

mask = ds_sorted < test_start_ts
cutoff_date = ds_sorted[mask].iloc[-1]
c = int(ds_sorted[ds_sorted == cutoff_date].index[0])
N = len(ds_sorted)

n_windows = int((((N - 1 - H) - c) // STEP_SIZE) + 1)

bkt_df = sf.cross_validation(
    df=ts,
    h=H,
    step_size=STEP_SIZE,
    n_windows=n_windows,
    prediction_intervals=ci,
    level=LEVELS,
)

bkt_df = bkt_df[bkt_df["ds"] >= test_start_ts].reset_index(drop=True)

In [40]:
# ----------------------------
# 5) Reusable output table (AutoARIMA) + plot-ready (1 forecast par date)
# ----------------------------
prefix = "autoarima"

model_col = [c for c in bkt_df.columns if c.lower().startswith(prefix)][0]
lo_col = [c for c in bkt_df.columns if c.lower().startswith(prefix) and c.lower().endswith("lo-95")][0]
hi_col = [c for c in bkt_df.columns if c.lower().startswith(prefix) and c.lower().endswith("hi-95")][0]

df_ar_forecasts = (
    bkt_df[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat",
        lo_col: "y_hat_lo_95",
        hi_col: "y_hat_hi_95",
    })
    .assign(
        date=lambda d: pd.to_datetime(d["date"]).dt.tz_localize(None),
        cutoff=lambda d: pd.to_datetime(d["cutoff"]).dt.tz_localize(None),
    )
    # IMPORTANT: 1 forecast par date (sinon graph faux)
    .sort_values(["series_id", "date", "cutoff"])
    .groupby(["series_id", "date"], as_index=False)
    .tail(1)  # garde le cutoff le plus récent
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

df_ar_forecasts

,series_id,date,cutoff,y_obs,y_hat,y_hat_lo_95,y_hat_hi_95
0,UNRATE,1991-10-01,1991-09-01,1.1,0.962727,0.756635,1.168819
1,UNRATE,1991-11-01,1991-09-01,0.8,0.926844,0.687007,1.166680
2,UNRATE,1991-12-01,1991-09-01,1.0,0.892298,0.546856,1.237739
3,UNRATE,1992-01-01,1991-09-01,0.9,0.859039,0.415294,1.302784
4,UNRATE,1992-02-01,1991-09-01,0.8,0.827020,0.068924,1.585117
...,...,...,...,...,...,...,...
139,UNRATE,2025-05-01,2024-09-01,0.2,0.103267,-0.912500,1.119034
140,UNRATE,2025-06-01,2024-09-01,0.0,0.090934,-1.125876,1.307743
141,UNRATE,2025-07-01,2024-09-01,0.0,0.080073,-0.916550,1.076696
142,UNRATE,2025-08-01,2024-09-01,0.1,0.070510,-0.643125,0.784145


# Graphique

In [41]:
# %% [Graph 1] df_obs (observations)
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [42]:
# %% [Graph 2] df_fcst (forecast + PI)
df_fcst = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat": "AR",
        "y_hat_lo_95": "AR-lo-95",
        "y_hat_hi_95": "AR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "AR",
        "AR-lo-95",
        "AR-hi-95",
    ]]
)

In [43]:
# %% [Graph 3] Plot
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (%)"
    elif trace.name == "AR":
        trace.name = "AutoARIMA (AR(p) updated every 36m)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

# Vérification

In [44]:
tmp = (bkt_df[["unique_id","ds","y"]]
       .drop_duplicates()
       .merge(ts[["unique_id","ds","y"]], on=["unique_id","ds"], how="inner", suffixes=("_cv","_src")))

print((tmp["y_cv"] - tmp["y_src"]).abs().describe())

count    144.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
dtype: float64


In [45]:
dups = ts.duplicated(["unique_id","ds"]).sum()
print("Doublons (unique_id, ds):", dups)

Doublons (unique_id, ds): 0


In [46]:
test_start_ts = pd.Timestamp(TEST_START, tz="UTC")

In [47]:
print(ts["y"].describe())
print(ts.tail(5)[["ds","y"]])

count    789.000000
mean      -0.019392
std        1.355527
min       -8.700000
25%       -0.600000
50%       -0.300000
75%        0.300000
max       11.100000
Name: y, dtype: float64
                           ds    y
784 2025-05-01 00:00:00+00:00  0.2
785 2025-06-01 00:00:00+00:00  0.0
786 2025-07-01 00:00:00+00:00  0.0
787 2025-08-01 00:00:00+00:00  0.1
788 2025-09-01 00:00:00+00:00  0.3


In [48]:
FEATURE_REFS_RAW = ["raw_value:value"]  # exemple: adapte au nom réel chez toi
ts_raw2 = load_features_from_feast(entity_df, FEATURE_REFS_RAW)

key_cols = {"series_id","date"}
feat_col = [c for c in ts_raw2.columns if c not in key_cols][0]

ts2 = (ts_raw2.rename(columns={"series_id":"unique_id","date":"ds", feat_col:"y_raw"})
              .sort_values(["unique_id","ds"])
              .reset_index(drop=True))

cmp = ts.merge(ts2, on=["unique_id","ds"], how="inner")
print((cmp["y"] - cmp["y_raw"]).abs().describe())
print(cmp.tail(5)[["ds","y","y_raw"]])

FeatureViewNotFoundException: Feature view raw_value does not exist in project unemployment_feature_store